# Deploying a trained model

_PyTorch_

**Take the trained weights out of the notebook: export to a Python-free runtime (TorchScript / ONNX), quantize to int8, and serve it behind an API.**

A trained model in a notebook is the start of the work, not the end. The notebook gives you weights that work; production demands a graph that is fast, portable, observable, and cheap to run.

       The core move is export: freeze the model into a standalone graph that no longer needs Python or the training code. TorchScript freezes it into a PyTorch-native serialized graph; ONNX freezes it into a framework-neutral file that other engines can run. Either way, the result is a single artifact you can drop onto a server, a phone, or a C++ binary.

---

This notebook walks the export path one step at a time — TorchScript, ONNX, int8 quantization, and a serving sketch — verifying the output matches at every boundary. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q onnx onnxruntime
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build the trained model and capture a reference output

Define the tiny image classifier and immediately call `.eval()` — the must-do before any export, so dropout/batch-norm behave deterministically. We push one example through under `torch.inference_mode()` (no autograd graph at serving) and save its output as `ref`. Every export below is checked against this reference so we know the converted graph still produces the same numbers.

In [ ]:
import torch, torch.nn as nn, numpy as np, os

torch.manual_seed(0)

# ---- a tiny image classifier (3x32x32 -> 10 classes) ----
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 8 * 8, 128)
        self.fc2   = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 16x16
        x = self.pool(torch.relu(self.conv2(x)))   #  8x8
        x = x.flatten(1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)                          # logits

model = Net().eval()                                # <-- inference mode (the famous must-do)
n_params = sum(p.numel() for p in model.parameters())
print("params:", n_params, "| fp32 MB:", round(n_params * 4 / 1e6, 3))

x = torch.randn(1, 3, 32, 32)                       # one example input
with torch.inference_mode():                        # no autograd graph at serving
    ref = model(x)

### Step 2 — Export to TorchScript and reload without the class

TorchScript freezes the model into a PyTorch-native serialized graph. `trace` records the ops for one example input (fast, but bakes in control flow), while `script` compiles the Python so data-dependent branches survive. We save the traced graph, reload it *without the `Net` class in scope*, and verify it matches the reference — proving the artifact is self-contained.

In [ ]:
# TORCHSCRIPT: trace (straight-line) and script (keeps control flow).
ts_traced = torch.jit.trace(model, x)               # records ops for THIS input
ts_scripted = torch.jit.script(model)               # compiles the Python -> graph
ts_traced.save("model_ts.pt")

reloaded = torch.jit.load("model_ts.pt")            # runs WITHOUT the Net class / Python
with torch.inference_mode():
    print("TorchScript matches PyTorch:",
          torch.allclose(reloaded(x), ref, atol=1e-5))

### Step 3 — Export to ONNX and run it with onnxruntime

ONNX freezes the model into a framework-neutral file that other engines can run. We pin the opset version (so the file stays loadable across runtime versions) and mark the batch dimension dynamic. After a structural validity check we run it through ONNX Runtime — the actual production path, with no PyTorch needed — and always verify the output matches across the boundary.

In [ ]:
# ONNX (Open Neural Network Exchange): export + run with onnxruntime.
# (the notebook setup cell installs onnx + onnxruntime)
import onnx, onnxruntime as ort

torch.onnx.export(
    model, x, "model.onnx",
    input_names=["input"], output_names=["logits"],
    opset_version=17,                               # PIN the opset across the boundary
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},  # variable batch size
)
onnx.checker.check_model(onnx.load("model.onnx"))   # structural validity

sess = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
onnx_out = sess.run(["logits"], {"input": x.numpy()})[0]
print("ONNX Runtime matches PyTorch:",
      np.allclose(onnx_out, ref.numpy(), atol=1e-4))   # ALWAYS verify across the boundary

### Step 4 — Quantize to int8, then sketch the serving endpoint

Dynamic quantization stores the Linear weights as int8 (1 byte) instead of float32 (4 bytes) — about 4x smaller and faster, but always re-validate accuracy before shipping. Finally we sketch a minimal FastAPI `/predict`: load the exported graph, reuse the *same* preprocessing as training (so there is no train/serve skew), and run each request under `inference_mode`.

In [ ]:
# DYNAMIC QUANTIZATION: int8 the Linear layers -> smaller + faster.
qmodel = torch.ao.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
).eval()
torch.save(model.state_dict(),  "fp32.pt")
torch.save(qmodel.state_dict(), "int8.pt")
print("fp32 file KB:", round(os.path.getsize("fp32.pt") / 1e3, 1),
      "| int8 file KB:", round(os.path.getsize("int8.pt") / 1e3, 1))
# NOTE: always re-validate accuracy on a held-out set before shipping int8!

# SERVE: a minimal FastAPI /predict sketch (preprocess shared w/ training).
SERVE = """
from fastapi import FastAPI
import torch
app = FastAPI()
model = torch.jit.load("model_ts.pt").eval()      # load the exported graph, eval mode

def preprocess(payload):                           # SAME steps as training -> no skew
    t = torch.tensor(payload["pixels"]).float().reshape(1, 3, 32, 32)
    return (t - 0.5) / 0.5                          # match the training normalization

@app.post("/predict")
def predict(payload: dict):
    x = preprocess(payload)
    with torch.inference_mode():                   # no grad graph per request
        logits = model(x)
    return {"class": int(logits.argmax(1).item())}
"""
print(SERVE)

## Visualize the data & results

_How much smaller and faster does the model get as you move from FP32 eager PyTorch to TorchScript, to ONNX Runtime, to int8 quantization?_

### Step 1 — Compute model size from the real architecture

Size on disk is just the parameter count times the bytes per weight. We add up the params layer by layer from the actual `Net` above, then convert: float32 is 4 bytes per weight, int8 is 1 byte. That is why int8 quantization gives the ~4x size reduction shown here.

In [ ]:
import numpy as np

# ---- model param count, layer by layer (real architecture from the CODE cell) ----
def conv(cin, cout, k):  return cout * cin * k * k + cout
def lin(fin, fout):      return fin * fout + fout

params = (
    conv(3, 16, 3)          # conv1 -> 448
  + conv(16, 32, 3)         # conv2 -> 4640
  + lin(32 * 8 * 8, 128)    # fc1   -> 262272
  + lin(128, 10)            # fc2   -> 1290
)
print("total params:", params)                      # 268650

# ---- size from param count: float32 = 4 bytes, int8 = 1 byte ----
fp32_mb = params * 4 / 1e6
int8_mb = params * 1 / 1e6
sizes = {
    "FP32 (.pt)":     round(fp32_mb, 3),            # 1.075
    "TorchScript":    round(fp32_mb * 1.01, 3),     # ~same weights + tiny graph overhead
    "ONNX Runtime":   round(fp32_mb, 3),            # same float32 weights
    "int8 quantized": round(int8_mb, 3),            # ~4x smaller -> 0.269
}
print("size MB:", sizes)
print("size reduction (int8):", round(fp32_mb / int8_mb, 2), "x")  # 4.0x

### Step 2 — Tabulate the illustrative latency

These latencies are illustrative numbers that show the typical ordering: eager PyTorch is slowest, TorchScript trims Python overhead, ONNX Runtime optimizes the graph further, and int8 quantization is fastest. The point is the *direction* of each export step, not exact milliseconds on your hardware.

In [ ]:
# ---- latency: illustrative, monotonically improving across optimized runtimes ----
latency_ms = {
    "FP32 eager":     12.0,
    "TorchScript":     9.0,
    "ONNX Runtime":    6.5,
    "int8 quantized":  4.0,
}
print("latency ms:", latency_ms)
print("speedup (int8 vs eager):", round(12.0 / 4.0, 2), "x")        # 3.0x

### Step 3 — Plot size and latency side by side

Two bar charts make the trade-off concrete: the left shows model size on disk (int8 visibly smaller), the right shows inference latency (lower is better, int8 fastest). Together they summarize why each export and quantization step is worth doing for production serving.

In [ ]:
# ---- charts ----
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cols = ["#ff7b72", "#4ea1ff", "#c89bff", "#7ee787"]

ax[0].bar(list(sizes.keys()), list(sizes.values()), color=cols)
ax[0].set_ylabel("size (MB)"); ax[0].set_title("model size on disk")
ax[0].tick_params(axis="x", rotation=20)

ax[1].bar(list(latency_ms.keys()), list(latency_ms.values()), color=cols)
ax[1].set_ylabel("latency (ms)"); ax[1].set_title("inference latency (lower is better)")
ax[1].tick_params(axis="x", rotation=20)

plt.tight_layout(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Build a tiny nn.Linear(4, 3) model, put it in model.eval(), trace it with torch.jit.trace(model, example) using an example of shape (1, 4), and confirm the TorchScript output matches the eager output with torch.allclose. Use torch.manual_seed(0).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call model.eval() before exporting, then torch.jit.trace(model, example). — _eval() is the must-do before any export; trace records the ops that run on the example input._
- Compare ts(x) to model(x) with torch.allclose. — _Always verify the exported graph reproduces the original numerically before trusting it._

**Answer:** import torch
import torch.nn as nn

torch.manual_seed(0)
model = nn.Linear(4, 3).eval()      # eval() before export -- the must-do
example = torch.randn(1, 4)
ts = torch.jit.trace(model, example)
with torch.inference_mode():
    print(torch.allclose(ts(example), model(example)))   # True

</details>

**Problem 2.** Type this in Colab. Demonstrate the trace-vs-script control-flow gotcha. Write a module whose forward returns x.sum() if x.sum() > 0 else -x.sum(). Trace it on a POSITIVE example, then feed it a NEGATIVE input and show the traced output is wrong. Re-export with torch.jit.script and show it is now correct.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Trace on a positive example, then call the traced module on a negative input. — _trace freezes whichever branch the example took, so the if is baked to one side for all future inputs._
- Re-export with torch.jit.script(m) and call it on the same negative input. — _script compiles the Python control flow, so the data-dependent if survives._

**Answer:** import torch
import torch.nn as nn

class M(nn.Module):
    def forward(self, x):
        if x.sum() > 0:
            return x.sum()
        return -x.sum()

m = M().eval()
traced = torch.jit.trace(m, torch.tensor([1.0, 2.0]))   # positive example
print(traced(torch.tensor([-5.0])).item())   # -5.0  (WRONG: branch frozen)

scripted = torch.jit.script(m)
print(scripted(torch.tensor([-5.0])).item()) #  5.0  (correct: if preserved)

</details>

**Problem 3.** Type this in Colab. Save a TorchScript module to disk and reload it WITHOUT the original class. Trace an nn.Linear(4, 2), call ts.save("m.ts"), then reloaded = torch.jit.load("m.ts"), and confirm reloaded produces the same output on a (3, 4) input. Predict the output shape first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Persist with ts.save("m.ts") and bring it back with torch.jit.load. — _A saved TorchScript graph carries its own ops + weights, so it runs without the Python model class &mdash; the whole point of exporting._
- Run a (3, 4) batch through the reloaded module and check the shape. — _A Linear(4, 2) maps the last dim 4&rarr;2, so a batch of 3 gives (3, 2)._

**Answer:** import torch
import torch.nn as nn

torch.manual_seed(0)
model = nn.Linear(4, 2).eval()
ts = torch.jit.trace(model, torch.randn(1, 4))
ts.save("m.ts")

reloaded = torch.jit.load("m.ts")     # no Net class needed
x = torch.randn(3, 4)
print(reloaded(x).shape)              # torch.Size([3, 2])
print(torch.allclose(reloaded(x), model(x)))   # True

</details>

**Problem 4.** Type this in Colab. Export a model to ONNX and verify the file exists. Build nn.Linear(8, 4).eval(), call torch.onnx.export with an example of shape (1, 8), opset_version=17, and dynamic_axes on the batch dimension, then use os.path.exists("m.onnx") and os.path.getsize to confirm the file was written.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call torch.onnx.export(model, example, "m.onnx", opset_version=17, dynamic_axes=...). — _Exporting traces the model into a portable ONNX graph; pinning the opset keeps it loadable across runtime versions._
- Check os.path.exists("m.onnx") and the byte size. — _Confirms the export actually produced an artifact on disk._

**Answer:** import torch, torch.nn as nn, os

model = nn.Linear(8, 4).eval()
x = torch.randn(1, 8)
torch.onnx.export(
    model, x, "m.onnx",
    input_names=["input"], output_names=["out"],
    opset_version=17,
    dynamic_axes={"input": {0: "batch"}, "out": {0: "batch"}},
)
print(os.path.exists("m.onnx"))           # True
print(os.path.getsize("m.onnx") > 0)      # True

</details>

**Problem 5.** Type this in Colab. Run an exported ONNX model with onnxruntime and check it matches PyTorch across the boundary. Export nn.Linear(8, 4).eval() to m.onnx, create an InferenceSession, run a (2, 8) input through it, and assert np.allclose(onnx_out, torch_out, atol=1e-4). (onnxruntime is auto-installed by the notebook setup cell.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Create ort.InferenceSession("m.onnx") and call .run(...) with the numpy input. — _ONNX Runtime executes the portable graph with no PyTorch installed &mdash; the production path._
- Compare the ONNX output to PyTorch's with np.allclose. — _Numeric drift across the ONNX boundary is a classic deployment bug; verify before shipping._

**Answer:** import torch, torch.nn as nn, numpy as np
import onnxruntime as ort

torch.manual_seed(0)
model = nn.Linear(8, 4).eval()
x = torch.randn(2, 8)
torch.onnx.export(model, x, "m.onnx",
                  input_names=["input"], output_names=["out"],
                  opset_version=17,
                  dynamic_axes={"input": {0: "batch"}, "out": {0: "batch"}})

with torch.inference_mode():
    torch_out = model(x).numpy()
sess = ort.InferenceSession("m.onnx", providers=["CPUExecutionProvider"])
onnx_out = sess.run(["out"], {"input": x.numpy()})[0]
print(np.allclose(onnx_out, torch_out, atol=1e-4))   # True

</details>

**Problem 6.** Type this in Colab. Apply dynamic int8 quantization to the linear layers and measure the file-size drop. Build a model with two nn.Linear layers, quantize with torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8), save both state dicts, and print the float32 vs int8 file sizes in KB.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8). — _Dynamic quantization stores Linear weights as int8 (1 byte) instead of float32 (4 bytes) &mdash; about 4&times; smaller._
- Save both state_dicts and compare os.path.getsize. — _Showing the real byte sizes makes the ~4&times; shrink concrete._

**Answer:** import torch, torch.nn as nn, os

model = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 64)).eval()
qmodel = torch.ao.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8).eval()

torch.save(model.state_dict(),  "fp32.pt")
torch.save(qmodel.state_dict(), "int8.pt")
print("fp32 KB:", round(os.path.getsize("fp32.pt") / 1e3, 1))   # ~ 345.x
print("int8 KB:", round(os.path.getsize("int8.pt") / 1e3, 1))   # ~  92.x (much smaller)
# NOTE: always re-validate accuracy on a held-out set before shipping int8!

</details>

**Problem 7.** Type this in Colab. Save a whole TorchScript MODULE (not just a state_dict) and reload it for serving. Trace a 2-layer model, save it with ts.save("served.ts"), load it with torch.jit.load into eval() mode, and run one prediction inside torch.inference_mode(), printing the predicted class via argmax.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Save the TorchScript module and reload it, then call .eval(). — _Serving loads the self-contained graph; eval() is non-negotiable so dropout/batch-norm behave deterministically._
- Run the prediction inside torch.inference_mode() and take argmax. — _inference_mode skips autograd bookkeeping you never use at serving, cutting per-request memory and latency._

**Answer:** import torch, torch.nn as nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 3)).eval()
ts = torch.jit.trace(model, torch.randn(1, 8))
ts.save("served.ts")

served = torch.jit.load("served.ts").eval()   # eval() for correct serving
x = torch.randn(1, 8)
with torch.inference_mode():                   # no autograd graph per request
    logits = served(x)
print(logits.argmax(1).item())                 # 2  (the predicted class index)

</details>

**Problem 8.** Type this in Colab. Show why model.eval() matters at serving. Build nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5)). With the model in train() mode, run the SAME input twice and print whether the two outputs match (they won't, because dropout is random). Switch to eval() and show the two outputs now match. Use torch.manual_seed(0).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- In train() mode, call the model twice on one input and compare with torch.equal. — _Dropout randomly zeroes units in training mode, so the same input gives different, non-deterministic answers &mdash; the famous serving bug._
- Call model.eval() and repeat. — _eval() turns dropout off (and batch-norm to running stats), making predictions deterministic._

**Answer:** import torch, torch.nn as nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5))
x = torch.randn(1, 4)

model.train()
print(torch.equal(model(x), model(x)))   # False  -- dropout is random!

model.eval()
print(torch.equal(model(x), model(x)))   # True   -- deterministic at serving

</details>